# InstaNovo + InstaNovo+ de novo predictions — Wastewater with FINETUNED models

Mirrors `instanovo_colab_wastewater.ipynb` but uses the fine-tuned
checkpoints (`model_finetune/{instanovo,instanovoplus}/model_best.ckpt`,
outputs of the finetune notebooks) — same shape as
`instanovo_colab_ecoli_finetune.ipynb` but for wastewater.

Runs both stages of the InstaNovo pipeline on all 4 wastewater fractions
(`wastewater_Sample{1,2}_{1,2}`):
1. InstaNovo (transformer) predicts directly on the spectra.
2. InstaNovo+ (diffusion) refines those predictions, using
   `refinement_path=` pointing at the InstaNovo CSVs from stage 1.

Outputs 24 CSVs total (3 input forms × 2 tools × 4 fractions) for the
Jetson-side FDR pipeline. Dir naming matches `run_conversions_finetune.sh`
and `merge_samples_finetune.sh` — `_finetune` comes BEFORE `_mgf` /
`_mgf_decoy`:

InstaNovo:
- `result_finetune/instanovo/wastewater/wastewater_Sample{1,2}_{1,2}.csv`             (mzML)
- `result_finetune_mgf/instanovo/wastewater/wastewater_Sample{1,2}_{1,2}.csv`         (MGF)
- `result_finetune_mgf_decoy/instanovo/wastewater/wastewater_Sample{1,2}_{1,2}.decoy.csv` (decoy MGF)

InstaNovo+:
- `result_finetune/instanovoplus/wastewater/wastewater_Sample{1,2}_{1,2}.csv`             (mzML)
- `result_finetune_mgf/instanovoplus/wastewater/wastewater_Sample{1,2}_{1,2}.csv`         (MGF)
- `result_finetune_mgf_decoy/instanovoplus/wastewater/wastewater_Sample{1,2}_{1,2}.decoy.csv` (decoy MGF)

Wastewater fractions get merged into per-sample CSVs by
`merge_samples_finetune.sh` on the Jetson side
(`wastewater_Sample{1,2}_{1,2}` → `wastewater_Sample{1,2}.csv`).

In [10]:
!nvidia-smi

Sat May  2 19:27:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Install dependencies

In [11]:
try:
  import instanovo
  !instanovo version
except ImportError:
  !pip install "instanovo[cu126]>=1.2.2" pyopenms-viz
  print('Installation complete. Restarting runtime to apply changes...')
  import os
  os.kill(os.getpid(), 9)

┏━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Package    ┃ Version ┃
┡━━━━━━━━━━━━╇━━━━━━━━━┩
│ InstaNovo  │ 1.2.2   │
│ InstaNovo+ │ 1.2.2   │
│ NumPy      │ 2.2.6   │
│ PyTorch    │ 2.8.0   │
└────────────┴─────────┘


## Sync inputs from Drive

In [12]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/data/wastewater
!mkdir -p /content/data_mgf/wastewater
!mkdir -p /content/data_mgf_decoy/wastewater
!mkdir -p /content/model_finetune/instanovo
!mkdir -p /content/model_finetune/instanovoplus
!mkdir -p /content/result_finetune/instanovo/wastewater
!mkdir -p /content/result_finetune_mgf/instanovo/wastewater
!mkdir -p /content/result_finetune_mgf_decoy/instanovo/wastewater
!mkdir -p /content/result_finetune/instanovoplus/wastewater
!mkdir -p /content/result_finetune_mgf/instanovoplus/wastewater
!mkdir -p /content/result_finetune_mgf_decoy/instanovoplus/wastewater

!mkdir -p /content/drive/MyDrive/DL-Project/result_finetune/instanovo/wastewater
!mkdir -p /content/drive/MyDrive/DL-Project/result_finetune_mgf/instanovo/wastewater
!mkdir -p /content/drive/MyDrive/DL-Project/result_finetune_mgf_decoy/instanovo/wastewater
!mkdir -p /content/drive/MyDrive/DL-Project/result_finetune/instanovoplus/wastewater
!mkdir -p /content/drive/MyDrive/DL-Project/result_finetune_mgf/instanovoplus/wastewater
!mkdir -p /content/drive/MyDrive/DL-Project/result_finetune_mgf_decoy/instanovoplus/wastewater

# Inputs — all 4 wastewater fractions
!cp -r /content/drive/MyDrive/DL-Project/data/wastewater/. /content/data/wastewater/
!cp -r /content/drive/MyDrive/DL-Project/data_mgf/wastewater/. /content/data_mgf/wastewater/
!cp -r /content/drive/MyDrive/DL-Project/data_mgf_decoy/wastewater/. /content/data_mgf_decoy/wastewater/

# Fine-tuned ckpts (outputs of the finetune notebooks)
!cp /content/drive/MyDrive/DL-Project/model_finetune/instanovo/model_best.ckpt    /content/model_finetune/instanovo/
!cp /content/drive/MyDrive/DL-Project/model_finetune/instanovoplus/model_best.ckpt /content/model_finetune/instanovoplus/

!ls -lh /content/data/wastewater/
!ls -lh /content/data_mgf/wastewater/
!ls -lh /content/data_mgf_decoy/wastewater/
!ls -lh /content/model_finetune/instanovo/
!ls -lh /content/model_finetune/instanovoplus/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 2.1G
-rw------- 1 root root 526M May  2 19:27 wastewater_Sample1_1.mzML
-rw------- 1 root root 515M May  2 19:27 wastewater_Sample1_2.mzML
-rw------- 1 root root 548M May  2 19:27 wastewater_Sample2_1.mzML
-rw------- 1 root root 540M May  2 19:27 wastewater_Sample2_2.mzML
total 519M
-rw------- 1 root root 149M May  2 19:28 wastewater_Sample1_1.mgf
-rw------- 1 root root 155M May  2 19:28 wastewater_Sample1_2.mgf
-rw------- 1 root root 137M May  2 19:28 wastewater_Sample2_1.mgf
-rw------- 1 root root  80M May  2 19:28 wastewater_Sample2_2.mgf
total 416M
-rw------- 1 root root 119M May  2 19:28 wastewater_Sample1_1.decoy.mgf
-rw------- 1 root root 125M May  2 19:28 wastewater_Sample1_2.decoy.mgf
-rw------- 1 root root 110M May  2 19:28 wastewater_Sample2_1.decoy.mgf
-rw------- 1 root root  64M May  2 19:28 wastewater_Sample2_2.decoy.mgf
total 724M
drwxr-x

Wastewater Instanovo (mzML)

In [13]:
import os

input_path = "/content/data/wastewater"
output_path = "/content/result_finetune/instanovo/wastewater"
model_path = "/content/model_finetune/instanovo/model_best.ckpt"
samples = ["wastewater_Sample1_1", "wastewater_Sample1_2", "wastewater_Sample2_1", "wastewater_Sample2_2"]

for sample in samples:
    in_file = f"{input_path}/{sample}.mzML"
    out_file = f"{output_path}/{sample}.csv"
    if os.path.exists(out_file):
        print(f"Skipping {sample} (already exists)")
        continue
    !instanovo transformer predict --data-path {in_file} --output-path {out_file} --instanovo-model {model_path} num_workers=4 batch_size=512

!cp -r /content/result_finetune/instanovo/wastewater/. /content/drive/MyDrive/DL-Project/result_finetune/instanovo/wastewater

[05/02/26 19:28:31] INFO     Initializing InstaNovo inference.                                                                                                                 
[05/02/26 19:28:33] INFO     NumExpr defaulting to 12 threads.                                                                                                                 
2026-05-02 19:28:36.552912: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 19:28:36.625179: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-05-02 19:28

Wastewater MGF Instanovo

In [14]:
import os

input_path = "/content/data_mgf/wastewater"
output_path = "/content/result_finetune_mgf/instanovo/wastewater"
model_path = "/content/model_finetune/instanovo/model_best.ckpt"
samples = ["wastewater_Sample1_1", "wastewater_Sample1_2", "wastewater_Sample2_1", "wastewater_Sample2_2"]

for sample in samples:
    in_file = f"{input_path}/{sample}.mgf"
    out_file = f"{output_path}/{sample}.csv"
    if os.path.exists(out_file):
        print(f"Skipping {sample} (already exists)")
        continue
    !instanovo transformer predict --data-path {in_file} --output-path {out_file} --instanovo-model {model_path} num_workers=4 batch_size=512

!cp -r /content/result_finetune_mgf/instanovo/wastewater/. /content/drive/MyDrive/DL-Project/result_finetune_mgf/instanovo/wastewater

[05/02/26 20:12:42] INFO     Initializing InstaNovo inference.                                                                                                                 
[05/02/26 20:12:44] INFO     NumExpr defaulting to 12 threads.                                                                                                                 
2026-05-02 20:12:47.305580: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 20:12:47.379543: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-05-02 20:12

Wastewater MGF Decoy Instanovo

In [15]:
import os

input_path = "/content/data_mgf_decoy/wastewater"
output_path = "/content/result_finetune_mgf_decoy/instanovo/wastewater"
model_path = "/content/model_finetune/instanovo/model_best.ckpt"
samples = ["wastewater_Sample1_1", "wastewater_Sample1_2", "wastewater_Sample2_1", "wastewater_Sample2_2"]

for sample in samples:
    in_file = f"{input_path}/{sample}.decoy.mgf"
    out_file = f"{output_path}/{sample}.decoy.csv"
    if os.path.exists(out_file):
        print(f"Skipping {sample} (already exists)")
        continue
    !instanovo transformer predict --data-path {in_file} --output-path {out_file} --instanovo-model {model_path} num_workers=4 batch_size=512

!cp -r /content/result_finetune_mgf_decoy/instanovo/wastewater/. /content/drive/MyDrive/DL-Project/result_finetune_mgf_decoy/instanovo/wastewater

[05/02/26 20:56:33] INFO     Initializing InstaNovo inference.                                                                                                                 
[05/02/26 20:56:35] INFO     NumExpr defaulting to 12 threads.                                                                                                                 
2026-05-02 20:56:38.192695: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 20:56:38.263846: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-05-02 20:56

Wastewater Instanovo+ (mzML)

Refines the InstaNovo mzML predictions with the fine-tuned InstaNovo+ ckpt.
`refinement_path=` points at the InstaNovo CSVs produced three cells above.

In [16]:
import os

input_path = "/content/data/wastewater"
output_path = "/content/result_finetune/instanovoplus/wastewater"
refinement_path = "/content/result_finetune/instanovo/wastewater"
model_path = "/content/model_finetune/instanovoplus/model_best.ckpt"
samples = ["wastewater_Sample1_1", "wastewater_Sample1_2", "wastewater_Sample2_1", "wastewater_Sample2_2"]

for sample in samples:
    in_file = f"{input_path}/{sample}.mzML"
    out_file = f"{output_path}/{sample}.csv"
    re_file = f"{refinement_path}/{sample}.csv"
    if os.path.exists(out_file):
        print(f"Skipping {sample} (already exists)")
        continue
    !instanovo diffusion predict --data-path {in_file} --output-path {out_file} --instanovo-plus-model {model_path} refinement_path={re_file} num_workers=4 batch_size=512

!cp -r /content/result_finetune/instanovoplus/wastewater/. /content/drive/MyDrive/DL-Project/result_finetune/instanovoplus/wastewater

[05/02/26 21:41:33] INFO     Initializing InstaNovo+ inference.                                                                                                                
[05/02/26 21:41:36] INFO     NumExpr defaulting to 12 threads.                                                                                                                 
2026-05-02 21:41:38.408277: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 21:41:38.478969: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-05-02 21:41

Wastewater MGF Instanovo+

Refines the InstaNovo MGF predictions with the fine-tuned InstaNovo+ ckpt.

In [17]:
import os

input_path = "/content/data_mgf/wastewater"
output_path = "/content/result_finetune_mgf/instanovoplus/wastewater"
refinement_path = "/content/result_finetune_mgf/instanovo/wastewater"
model_path = "/content/model_finetune/instanovoplus/model_best.ckpt"
samples = ["wastewater_Sample1_1", "wastewater_Sample1_2", "wastewater_Sample2_1", "wastewater_Sample2_2"]

for sample in samples:
    in_file = f"{input_path}/{sample}.mgf"
    out_file = f"{output_path}/{sample}.csv"
    re_file = f"{refinement_path}/{sample}.csv"
    if os.path.exists(out_file):
        print(f"Skipping {sample} (already exists)")
        continue
    !instanovo diffusion predict --data-path {in_file} --output-path {out_file} --instanovo-plus-model {model_path} refinement_path={re_file} num_workers=4 batch_size=512

!cp -r /content/result_finetune_mgf/instanovoplus/wastewater/. /content/drive/MyDrive/DL-Project/result_finetune_mgf/instanovoplus/wastewater

[05/02/26 22:05:17] INFO     Initializing InstaNovo+ inference.                                                                                                                
[05/02/26 22:05:19] INFO     NumExpr defaulting to 12 threads.                                                                                                                 
2026-05-02 22:05:22.057255: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 22:05:22.128497: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-05-02 22:05

Wastewater MGF Decoy Instanovo+

Refines the InstaNovo decoy-MGF predictions with the fine-tuned InstaNovo+ ckpt.

In [18]:
import os

input_path = "/content/data_mgf_decoy/wastewater"
output_path = "/content/result_finetune_mgf_decoy/instanovoplus/wastewater"
refinement_path = "/content/result_finetune_mgf_decoy/instanovo/wastewater"
model_path = "/content/model_finetune/instanovoplus/model_best.ckpt"
samples = ["wastewater_Sample1_1", "wastewater_Sample1_2", "wastewater_Sample2_1", "wastewater_Sample2_2"]

for sample in samples:
    in_file = f"{input_path}/{sample}.decoy.mgf"
    out_file = f"{output_path}/{sample}.decoy.csv"
    re_file = f"{refinement_path}/{sample}.decoy.csv"
    if os.path.exists(out_file):
        print(f"Skipping {sample} (already exists)")
        continue
    !instanovo diffusion predict --data-path {in_file} --output-path {out_file} --instanovo-plus-model {model_path} refinement_path={re_file} num_workers=4 batch_size=512

!cp -r /content/result_finetune_mgf_decoy/instanovoplus/wastewater/. /content/drive/MyDrive/DL-Project/result_finetune_mgf_decoy/instanovoplus/wastewater

[05/02/26 22:28:31] INFO     Initializing InstaNovo+ inference.                                                                                                                
[05/02/26 22:28:33] INFO     NumExpr defaulting to 12 threads.                                                                                                                 
2026-05-02 22:28:36.146394: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 22:28:36.217740: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-05-02 22:28